# SMETANA and SteadyCom

In [1]:
import reframed
from reframed import SteadyCom
import pandas as pd
import numpy as np
from os import listdir
import warnings
import seaborn as sns
from collections import OrderedDict
import json

import subprocess
import time

#### Load data and models

**Load relative abundance (E. coli)**

In [2]:
relative_abundance_ecoli = pd.read_csv("../data/Guillen2021/relative_abundance.csv",index_col=0)
relative_abundance_ecoli.index = relative_abundance_ecoli.index.map(lambda x:x if x!="yhhK" else "panZ")

**Load models**

*E. coli mutants  (top 7)*

In [3]:
RA_ecoli_mean = relative_abundance_ecoli.T.mean().sort_values(ascending=False)
top7 = RA_ecoli_mean[RA_ecoli_mean>0.01].index
top7_string = [f"mutant_{mutant}.xml" for mutant in top7]

In [4]:
models_ecoli = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Guillen2021/strains/"):
        if path not in top7_string: continue
        models_ecoli[path.split("_")[1][:-4]] = reframed.load_cbmodel("../models/Guillen2021/strains/"+path,
                                                        exchange_detection="^R_EX_(?!nad_c$|thf_c$).*")

*Gut microbiome members*

In [5]:
models_gut = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Venturelli2018/species_used_noO2/"):
        models_gut[path[:-4]] = reframed.load_cbmodel("../models/Venturelli2018/species_used_noO2/"+path,
                                                      exchange_detection="R_EX_(?!biomass__40__e__41__)")


**Load media**

*For E. coli mutants (M9 medium to force growth of specific members)*

In [6]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = list(media_db[media_db.medium=="M9"].compound.values)
env_ecoli = reframed.Environment.from_compounds(compounds,max_uptake=10)

*For gut microbiome members*

In [7]:
media_db_gut = pd.read_csv("../data/Venturelli2018/generated/media_db.tsv",sep="\t")
reactions = media_db_gut[media_db_gut.medium=="LB_extend"].compound.values
#reactions = [rxn.replace("40","91").replace("41","93") for rxn in reactions]

env_gut = reframed.Environment.from_reactions(reactions,max_uptake=10)

**Load other data**

In [8]:
gene_id2name = pd.read_csv("../data/Guillen2021/generated/gene_id2name.tsv",sep="\t")

## SMETANA

### Prepare files for SMETANA 

We need:
- Community models
- Community data file

We also need:
- Media



**E.coli mutants**

*Prepare files*

**Gut microbiome**

*Prepare files*

### Run SMETANA

## SteadyCom

In [9]:
reframed.set_default_solver("cplex")

### Run SteadyCom for E.coli 

In [10]:
comm_ecoli = reframed.Community(community_id="E.coli",models=models_ecoli.values())

env_ecoli.apply(comm_ecoli.merged_model,inplace=True,exclusive=True)

sol_steadycom_ecoli = SteadyCom(comm_ecoli)

In [22]:
cross_feeding_ecoli = pd.DataFrame(sol_steadycom_ecoli.cross_feeding())
cross_feeding_ecoli.columns = ["donor","receiver","compound","rate?"]

relative_abundance_ecoli = {"relative_abundance":sol_steadycom_ecoli.abundance}
growth_ecoli = {"growth":{"E.coli mutants":sol_steadycom_ecoli.growth}}

In [24]:
pd.DataFrame(growth_ecoli).to_csv("../output/Guillen2021/steadycom/growth.tsv",sep="\t")

pd.DataFrame(relative_abundance_ecoli).to_csv("../output/Guillen2021/steadycom/relative_abundance.tsv",sep="\t")

pd.DataFrame(cross_feeding_ecoli).to_csv("../output/Guillen2021/steadycom/cross_feeding.tsv",sep="\t",index=None)

### Run SteadyCom for Gut microbiome 

In [25]:
cross_feeding_gut_comms = {}
relative_abundance_gut_comms = {}
growth_gut_comms = {}

community_id = "Full"

start = time.time()
time_1 = time.time()
print(f"Started SteadyCom for {community_id} after: {time_1 - start:.4f} seconds")

comm_gut = reframed.Community(community_id=community_id,models=models_gut.values())
env_gut.apply(comm_gut.merged_model,inplace=True,exclusive=True)
sol_steadycom_gut = SteadyCom(comm_gut)

time_2 = time.time()
print(f"Running SteadyCom for {community_id} took: {time_2 - time_1:.4f} seconds")

cross_feeding_gut = pd.DataFrame(sol_steadycom_gut.cross_feeding())
cross_feeding_gut.columns = ["donor","receiver","compound","rate?"]
relative_abundance_gut = sol_steadycom_gut.abundance
growth_gut = sol_steadycom_gut.growth

time_3 = time.time()
print(f"Collecting results for {community_id} took: {time_3 - time_2:.4f} seconds")

cross_feeding_gut_comms["Full"] = cross_feeding_gut
relative_abundance_gut_comms["Full"] = relative_abundance_gut
growth_gut_comms["Full"] = growth_gut



for model_id, model in models_gut.items():
    start = time.time()
    models_gut_copy = models_gut.copy()
    models_gut_copy.pop(model_id)
    community_id = model_id.split("_")[0][0].upper()+model_id.split("_")[1][0].upper()+"-"
    
    time_1 = time.time()
    print(f"Started SteadyCom for {community_id} after: {time_1 - start:.4f} seconds")
    
    comm_gut_temp = reframed.Community(community_id=community_id,models=models_gut_copy.values())
    env_gut.apply(comm_gut_temp.merged_model,inplace=True,exclusive=True)
    sol_steadycom_gut_temp = SteadyCom(comm_gut_temp)
    
    time_2 = time.time()
    print(f"Running SteadyCom for {community_id} took: {time_2 - time_1:.4f} seconds")

    cross_feeding_gut = pd.DataFrame(sol_steadycom_gut_temp.cross_feeding())
    cross_feeding_gut.columns = ["donor","receiver","compound","rate?"]
    relative_abundance_gut = sol_steadycom_gut_temp.abundance
    growth_gut = sol_steadycom_gut_temp.growth
    
    time_3 = time.time()
    print(f"Collecting results for {community_id} took: {time_3 - time_2:.4f} seconds")

    
    cross_feeding_gut_comms[community_id] = cross_feeding_gut
    relative_abundance_gut_comms[community_id] = relative_abundance_gut
    growth_gut_comms[community_id] = growth_gut

Started SteadyCom for Full after: 0.0001 seconds
Running SteadyCom for Full took: 1200.0618 seconds
Started SteadyCom for BU- after: 0.0000 seconds
Running SteadyCom for BU- took: 747.6923 seconds
Started SteadyCom for BV- after: 0.0000 seconds
Running SteadyCom for BV- took: 220.8505 seconds
Started SteadyCom for CA- after: 0.0000 seconds
Running SteadyCom for CA- took: 177.1099 seconds
Started SteadyCom for FP- after: 0.0000 seconds
Running SteadyCom for FP- took: 336.7095 seconds
Started SteadyCom for EL- after: 0.0000 seconds


/Users/idunmariaburgos/opt/anaconda3/envs/steadiercom_preanalysis/lib/python3.8/site-packages/reframed/core/environment.py:75: UserWarning: Exchange reaction not in model: R_EX_lanost__91__e__93__
  warn(f'Exchange reaction not in model: {r_id}')


Running SteadyCom for EL- took: 157.6889 seconds
Started SteadyCom for PC- after: 0.0000 seconds
Running SteadyCom for PC- took: 359.0331 seconds
Started SteadyCom for CH- after: 0.0000 seconds
Running SteadyCom for CH- took: 522.9184 seconds
Started SteadyCom for ER- after: 0.0000 seconds


/Users/idunmariaburgos/opt/anaconda3/envs/steadiercom_preanalysis/lib/python3.8/site-packages/reframed/core/environment.py:75: UserWarning: Exchange reaction not in model: R_EX_asp_L__91__e__93__
  warn(f'Exchange reaction not in model: {r_id}')


Running SteadyCom for ER- took: 300.9572 seconds
Started SteadyCom for DP- after: 0.0000 seconds


/Users/idunmariaburgos/opt/anaconda3/envs/steadiercom_preanalysis/lib/python3.8/site-packages/reframed/core/environment.py:75: UserWarning: Exchange reaction not in model: R_EX_arab_D__91__e__93__
  warn(f'Exchange reaction not in model: {r_id}')


Running SteadyCom for DP- took: 181.4255 seconds
Started SteadyCom for BO- after: 0.0000 seconds
Running SteadyCom for BO- took: 421.5032 seconds
Started SteadyCom for BT- after: 0.0000 seconds
Running SteadyCom for BT- took: 69.2503 seconds
Started SteadyCom for BH- after: 0.0000 seconds


/Users/idunmariaburgos/opt/anaconda3/envs/steadiercom_preanalysis/lib/python3.8/site-packages/reframed/core/environment.py:75: UserWarning: Exchange reaction not in model: R_EX_nmn__91__e__93__
  warn(f'Exchange reaction not in model: {r_id}')


Running SteadyCom for BH- took: 254.8287 seconds
